# Practical Assignment 13: Image Classification using Transfer Learning on Animal Dataset

**Aim:** Implement image classification using transfer learning of CNN architectures on the Cats vs Dogs animal dataset.

**Key Concepts to Remember:**
1. **Feature Reuse:** MobileNetV2 pretrained on ImageNet captures rich edges, shapes, and textures.
2. **Freeze Backbone:** `base_model.trainable = False` prevents destruction of pretrained weights.
3. **Custom Top Head:** GlobalAveragePooling $\to$ Dense(1, sigmoid) for binary classification.


In [1]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
import tensorflow as tf
import numpy as np

# Step 1: Load Cats vs Dogs Dataset
train_dir = r'd:/workspace01/4th year/DL/LabPracticesIV_DEEP_LEARNING/dataset/cats_and_dogs_filtered/train'
val_dir = r'd:/workspace01/4th year/DL/LabPracticesIV_DEEP_LEARNING/dataset/cats_and_dogs_filtered/validation'

IMG_SIZE = (128, 128)
BATCH_SIZE = 32

train_ds = tf.keras.utils.image_dataset_from_directory(train_dir, image_size=IMG_SIZE, batch_size=BATCH_SIZE).take(25)
val_ds = tf.keras.utils.image_dataset_from_directory(val_dir, image_size=IMG_SIZE, batch_size=BATCH_SIZE).take(10)
class_names = ['cats', 'dogs']

# Step 2: Build Transfer Learning Model with Pretrained MobileNetV2
base_model = tf.keras.applications.MobileNetV2(input_shape=(128, 128, 3), include_top=False, weights='imagenet')
base_model.trainable = False

model = tf.keras.Sequential([
    tf.keras.layers.Rescaling(1./127.5, offset=-1, input_shape=(128, 128, 3)),  # MobileNetV2 scale [-1, 1]
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(1, activation='sigmoid')
])
model.summary()

# Step 3: Compile and Train Model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
history = model.fit(train_ds, epochs=2, validation_data=val_ds, verbose=1)

# Step 4: Evaluate Model
loss, acc = model.evaluate(val_ds, verbose=0)
print(f'\nTransfer Learning Validation Accuracy: {acc*100:.2f}% (High accuracy in 2 epochs due to ImageNet pretraining)')

for images, labels in val_ds.take(1):
    preds = model.predict(images[:3], verbose=0)
    for i in range(3):
        pred_c = class_names[int(preds[i][0] > 0.5)]
        true_c = class_names[labels[i]]
        print(f'Animal {i+1} -> Predicted: {pred_c:<6} | Actual: {true_c}')


Found 2000 files belonging to 2 classes.
Found 1000 files belonging to 2 classes.
Model: "sequential_10"
┌─────────────────────────────────┬────────────────────────┬───────────────┐
│ Layer (type)                    │ Output Shape           │       Param # │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rescaling_3 (Rescaling)         │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_128            │ (None, 4, 4, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 1280)

C:\Users\Rihan\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\preprocessing\data_layer.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
